# Lesson 2b: Dynamic Programming Practical - Advanced Methods

**Reinforcement Learning Course - Powell Clark**

---

## Learning Objectives

By the end of this notebook, you will:

1. **Implement Asynchronous DP** - State update ordering and convergence
2. **Implement Prioritized Sweeping** - Focus computation on important states
3. **Implement Modified Policy Iteration** - Balance between PI and VI
4. **Implement In-Place Updates** - Memory-efficient value iteration
5. **Benchmark Performance** - Compare variants on multiple environments
6. **Analyze Complexity** - Empirical validation of theoretical predictions

This notebook provides advanced dynamic programming implementations that improve upon the basic methods from Lesson 1b.

---

## Table of Contents

1. [Setup and Utilities](#setup)
2. [Asynchronous Value Iteration](#async-vi)
3. [Prioritized Sweeping](#prioritized)
4. [Modified Policy Iteration](#modified-pi)
5. [In-Place Updates](#inplace)
6. [Performance Benchmarking](#benchmark)
7. [Complexity Analysis](#complexity)
8. [Visualizations](#visualization)
9. [Exercises](#exercises)

---

## 1. Setup and Utilities <a id='setup'></a>

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List, Optional, Set
from dataclasses import dataclass
from collections import defaultdict
import heapq
import time

# Gymnasium
import gymnasium as gym

# Configuration
np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✅ Imports successful")

In [ ]:
# MDP data structure (from Lesson 1b)
@dataclass
class MDP:
    """Markov Decision Process definition."""
    n_states: int
    n_actions: int
    P: np.ndarray  # Transition probabilities [s, a, s'] -> probability
    R: np.ndarray  # Rewards [s, a, s'] -> reward
    gamma: float   # Discount factor
    terminal_states: Optional[List[int]] = None
    
    def __post_init__(self):
        if self.terminal_states is None:
            self.terminal_states = []


def compute_action_values(mdp: MDP, V: np.ndarray, state: int) -> np.ndarray:
    """Compute Q(s, a) for all actions in a given state."""
    Q = np.zeros(mdp.n_actions)
    for a in range(mdp.n_actions):
        for s_prime in range(mdp.n_states):
            prob = mdp.P[state, a, s_prime]
            reward = mdp.R[state, a, s_prime]
            Q[a] += prob * (reward + mdp.gamma * V[s_prime])
    return Q


def extract_policy(mdp: MDP, V: np.ndarray) -> np.ndarray:
    """Extract greedy policy from value function."""
    policy = np.zeros((mdp.n_states, mdp.n_actions))
    for s in range(mdp.n_states):
        Q = compute_action_values(mdp, V, s)
        best_action = np.argmax(Q)
        policy[s, best_action] = 1.0
    return policy


def extract_mdp_from_gym(env: gym.Env, gamma: float = 0.99) -> MDP:
    """Extract MDP structure from Gymnasium environment."""
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    
    P = np.zeros((n_states, n_actions, n_states))
    R = np.zeros((n_states, n_actions, n_states))
    
    for state in range(n_states):
        for action in range(n_actions):
            transitions = env.unwrapped.P[state][action]
            for prob, next_state, reward, done in transitions:
                P[state, action, next_state] += prob
                R[state, action, next_state] = reward
    
    terminal_states = []
    for s in range(n_states):
        is_terminal = all(np.isclose(P[s, a, s], 1.0) for a in range(n_actions))
        if is_terminal:
            terminal_states.append(s)
    
    return MDP(n_states, n_actions, P, R, gamma, terminal_states)


print("✅ Utility functions defined")

In [ ]:
# Load test environment
env = gym.make('FrozenLake-v1', is_slippery=True, render_mode=None)
test_mdp = extract_mdp_from_gym(env)

print(f"Test MDP: {test_mdp.n_states} states, {test_mdp.n_actions} actions")
print(f"Terminal states: {test_mdp.terminal_states}")
print("✅ Test environment loaded")

## 2. Asynchronous Value Iteration <a id='async-vi'></a>

### Theory

In **asynchronous value iteration**, states are updated in any order (not necessarily sequentially). The algorithm still converges to $V^*$ as long as all states continue to be updated.

**Advantages**:
- More flexible computation scheduling
- Can prioritize important states
- Better for online/real-time applications

### Implementation

In [ ]:
def async_value_iteration(
    mdp: MDP,
    theta: float = 1e-6,
    max_iterations: int = 10000,
    state_order: str = 'random'
) -> Tuple[np.ndarray, np.ndarray, int, List[float]]:
    """
    Asynchronous value iteration with configurable state ordering.
    
    Args:
        mdp: Markov Decision Process
        theta: Convergence threshold
        max_iterations: Maximum sweeps through state space
        state_order: 'sequential', 'random', or 'reverse'
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        iterations: Number of sweeps
        delta_history: Maximum value change per sweep
    """
    V = np.zeros(mdp.n_states)
    delta_history = []
    
    # Create state ordering
    states = list(range(mdp.n_states))
    
    for iteration in range(max_iterations):
        delta = 0
        
        # Determine order for this sweep
        if state_order == 'random':
            np.random.shuffle(states)
        elif state_order == 'reverse':
            states = list(reversed(states))
        # 'sequential' uses natural order
        
        # Update states in determined order
        for s in states:
            if s in mdp.terminal_states:
                continue
            
            v = V[s]
            Q = compute_action_values(mdp, V, s)
            V[s] = np.max(Q)
            delta = max(delta, abs(v - V[s]))
        
        delta_history.append(delta)
        
        if delta < theta:
            policy = extract_policy(mdp, V)
            return policy, V, iteration + 1, delta_history
    
    policy = extract_policy(mdp, V)
    return policy, V, max_iterations, delta_history


print("✅ Asynchronous value iteration implemented")

### Testing Asynchronous VI

In [ ]:
print("Comparing State Ordering Strategies")
print("=" * 60)

results = {}
for order in ['sequential', 'random', 'reverse']:
    start = time.time()
    policy, V, iters, deltas = async_value_iteration(test_mdp, state_order=order)
    elapsed = time.time() - start
    
    results[order] = {
        'iterations': iters,
        'time': elapsed,
        'V': V,
        'deltas': deltas
    }
    
    print(f"\n{order.capitalize()} order:")
    print(f"  Iterations: {iters}")
    print(f"  Time: {elapsed:.6f}s")
    print(f"  Final max V: {np.max(V):.6f}")

# Compare convergence
fig, ax = plt.subplots(figsize=(10, 6))
for order, data in results.items():
    ax.plot(data['deltas'], label=order.capitalize(), linewidth=2, alpha=0.8)

ax.set_xlabel('Sweep', fontsize=12)
ax.set_ylabel('Max Value Change (Δ)', fontsize=12)
ax.set_title('Asynchronous VI: State Ordering Comparison', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ Asynchronous VI comparison complete")

## 3. Prioritized Sweeping <a id='prioritized'></a>

### Theory

**Prioritized sweeping** updates states in order of their Bellman error magnitude:

$$p(s) = \left| V(s) - \max_a \sum_{s'} p(s'|s,a)[r + \gamma V(s')] \right|$$

States with larger errors are updated first, focusing computation where it matters most.

### Implementation

In [ ]:
class PriorityQueue:
    """Min-heap priority queue for prioritized sweeping."""
    
    def __init__(self):
        self.heap = []
        self.entry_map = {}  # state -> [priority, state]
        
    def push(self, state: int, priority: float):
        """Add or update state with given priority (negative for max-heap)."""
        if state in self.entry_map:
            # Mark old entry as removed
            old_entry = self.entry_map[state]
            old_entry[-1] = None  # Mark as removed
        
        # Add new entry (negate priority for max-heap behavior)
        entry = [-priority, state]
        self.entry_map[state] = entry
        heapq.heappush(self.heap, entry)
    
    def pop(self) -> Optional[Tuple[int, float]]:
        """Remove and return state with highest priority."""
        while self.heap:
            priority, state = heapq.heappop(self.heap)
            if state is not None:  # Not marked as removed
                del self.entry_map[state]
                return state, -priority
        return None
    
    def empty(self) -> bool:
        """Check if queue is empty."""
        return len(self.entry_map) == 0
    
    def __len__(self) -> int:
        return len(self.entry_map)


def build_predecessor_map(mdp: MDP) -> Dict[int, Set[int]]:
    """
    Build map of predecessor states.
    
    Returns:
        predecessors: Dict mapping state -> set of predecessor states
    """
    predecessors = defaultdict(set)
    
    for s in range(mdp.n_states):
        for a in range(mdp.n_actions):
            for s_prime in range(mdp.n_states):
                if mdp.P[s, a, s_prime] > 0:
                    predecessors[s_prime].add(s)
    
    return predecessors


def prioritized_sweeping(
    mdp: MDP,
    theta: float = 1e-6,
    min_priority: float = 1e-10,
    max_updates: int = 100000
) -> Tuple[np.ndarray, np.ndarray, int, List[float]]:
    """
    Prioritized sweeping value iteration.
    
    Args:
        mdp: Markov Decision Process
        theta: Convergence threshold
        min_priority: Minimum priority to update
        max_updates: Maximum number of state updates
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        updates: Number of state updates
        priority_history: Priority values over time
    """
    V = np.zeros(mdp.n_states)
    pqueue = PriorityQueue()
    predecessors = build_predecessor_map(mdp)
    priority_history = []
    
    # Initialize priorities for all states
    for s in range(mdp.n_states):
        if s not in mdp.terminal_states:
            Q = compute_action_values(mdp, V, s)
            priority = abs(V[s] - np.max(Q))
            if priority > min_priority:
                pqueue.push(s, priority)
    
    updates = 0
    max_delta = float('inf')
    
    while not pqueue.empty() and updates < max_updates and max_delta >= theta:
        # Get state with highest priority
        result = pqueue.pop()
        if result is None:
            break
        
        s, priority = result
        priority_history.append(priority)
        
        # Update state
        v_old = V[s]
        Q = compute_action_values(mdp, V, s)
        V[s] = np.max(Q)
        delta = abs(v_old - V[s])
        max_delta = delta
        
        updates += 1
        
        # Update priorities of predecessor states
        for s_pred in predecessors[s]:
            if s_pred not in mdp.terminal_states:
                Q_pred = compute_action_values(mdp, V, s_pred)
                priority_pred = abs(V[s_pred] - np.max(Q_pred))
                if priority_pred > min_priority:
                    pqueue.push(s_pred, priority_pred)
    
    policy = extract_policy(mdp, V)
    return policy, V, updates, priority_history


print("✅ Prioritized sweeping implemented")

### Testing Prioritized Sweeping

In [ ]:
print("Testing Prioritized Sweeping")
print("=" * 60)

start = time.time()
policy_ps, V_ps, updates_ps, priorities = prioritized_sweeping(test_mdp)
time_ps = time.time() - start

print(f"\nPrioritized Sweeping:")
print(f"  State updates: {updates_ps}")
print(f"  Time: {time_ps:.6f}s")
print(f"  Final max V: {np.max(V_ps):.6f}")

# Compare with standard VI
start = time.time()
policy_vi, V_vi, iters_vi, deltas_vi = async_value_iteration(test_mdp, state_order='sequential')
time_vi = time.time() - start

total_updates_vi = iters_vi * test_mdp.n_states

print(f"\nStandard Value Iteration:")
print(f"  Sweeps: {iters_vi}")
print(f"  Total state updates: {total_updates_vi}")
print(f"  Time: {time_vi:.6f}s")
print(f"  Final max V: {np.max(V_vi):.6f}")

print(f"\nComparison:")
print(f"  Update reduction: {100 * (1 - updates_ps / total_updates_vi):.1f}%")
print(f"  Value difference: {np.max(np.abs(V_ps - V_vi)):.2e}")

print("\n✅ Prioritized sweeping test complete")

### Priority Evolution Visualization

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(priorities, linewidth=2, alpha=0.7, color='purple')
ax.set_xlabel('State Update', fontsize=12)
ax.set_ylabel('Priority (Bellman Error)', fontsize=12)
ax.set_title('Prioritized Sweeping: Priority Evolution', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/2b_prioritized_sweeping.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Priority evolution visualization created")

## 4. Modified Policy Iteration <a id='modified-pi'></a>

### Theory

**Modified policy iteration** performs only $k$ sweeps of policy evaluation before improvement, interpolating between:
- $k = 1$: Value iteration
- $k = \infty$: Standard policy iteration

### Implementation

In [ ]:
def modified_policy_iteration(
    mdp: MDP,
    k: int = 10,
    theta: float = 1e-6,
    max_policy_iterations: int = 100
) -> Tuple[np.ndarray, np.ndarray, int, int]:
    """
    Modified policy iteration with k evaluation sweeps.
    
    Args:
        mdp: Markov Decision Process
        k: Number of evaluation sweeps per policy improvement
        theta: Convergence threshold for value changes
        max_policy_iterations: Maximum number of policy iterations
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        policy_iterations: Number of policy iterations
        total_sweeps: Total number of value sweeps
    """
    # Initialize with random policy
    policy = np.ones((mdp.n_states, mdp.n_actions)) / mdp.n_actions
    V = np.zeros(mdp.n_states)
    total_sweeps = 0
    
    for policy_iter in range(max_policy_iterations):
        # Policy Evaluation: k sweeps
        for sweep in range(k):
            delta = 0
            for s in range(mdp.n_states):
                if s in mdp.terminal_states:
                    continue
                
                v = V[s]
                
                # Bellman expectation update
                new_v = 0
                for a in range(mdp.n_actions):
                    Q_sa = sum(
                        mdp.P[s, a, s_prime] * (mdp.R[s, a, s_prime] + mdp.gamma * V[s_prime])
                        for s_prime in range(mdp.n_states)
                    )
                    new_v += policy[s, a] * Q_sa
                
                V[s] = new_v
                delta = max(delta, abs(v - V[s]))
            
            total_sweeps += 1
            
            # Early stopping if converged
            if delta < theta:
                break
        
        # Policy Improvement
        policy_stable = True
        new_policy = np.zeros_like(policy)
        
        for s in range(mdp.n_states):
            old_action = np.argmax(policy[s])
            Q = compute_action_values(mdp, V, s)
            new_action = np.argmax(Q)
            new_policy[s, new_action] = 1.0
            
            if new_action != old_action:
                policy_stable = False
        
        policy = new_policy
        
        if policy_stable:
            return policy, V, policy_iter + 1, total_sweeps
    
    return policy, V, max_policy_iterations, total_sweeps


print("✅ Modified policy iteration implemented")

### Testing Modified PI with Different k

In [ ]:
print("Testing Modified Policy Iteration with Different k")
print("=" * 60)

k_values = [1, 3, 5, 10, 20, 50]
mpi_results = []

for k in k_values:
    start = time.time()
    policy, V, pi_iters, total_sweeps = modified_policy_iteration(test_mdp, k=k)
    elapsed = time.time() - start
    
    mpi_results.append({
        'k': k,
        'policy_iters': pi_iters,
        'total_sweeps': total_sweeps,
        'time': elapsed,
        'V': V
    })
    
    print(f"\nk = {k:3d}:")
    print(f"  Policy iterations: {pi_iters}")
    print(f"  Total sweeps: {total_sweeps}")
    print(f"  Time: {elapsed:.6f}s")

print("\n✅ Modified PI testing complete")

### Visualize k Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

k_vals = [r['k'] for r in mpi_results]
pi_iters = [r['policy_iters'] for r in mpi_results]
sweeps = [r['total_sweeps'] for r in mpi_results]

# Policy iterations vs k
axes[0].plot(k_vals, pi_iters, marker='o', linewidth=2, markersize=8, color='blue')
axes[0].set_xlabel('k (evaluation sweeps)', fontsize=12)
axes[0].set_ylabel('Policy Iterations', fontsize=12)
axes[0].set_title('Modified PI: Policy Iterations vs k', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')

# Total sweeps vs k
axes[1].plot(k_vals, sweeps, marker='s', linewidth=2, markersize=8, color='green')
axes[1].set_xlabel('k (evaluation sweeps)', fontsize=12)
axes[1].set_ylabel('Total Value Sweeps', fontsize=12)
axes[1].set_title('Modified PI: Total Sweeps vs k', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/2b_modified_pi.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Modified PI visualization created")

## 5. In-Place Updates <a id='inplace'></a>

### Theory

**In-place updates** use the most recent value estimates immediately rather than waiting for a full sweep.

**Standard** (two arrays):
```python
V_new[s] = max_a Σ p(s'|s,a)[r + γ V_old[s']]
```

**In-place** (one array):
```python
V[s] = max_a Σ p(s'|s,a)[r + γ V[s']]  # Uses updated V immediately
```

### Implementation

In [ ]:
def value_iteration_inplace(
    mdp: MDP,
    theta: float = 1e-6,
    max_iterations: int = 10000
) -> Tuple[np.ndarray, np.ndarray, int, List[float]]:
    """
    Value iteration with in-place updates.
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        iterations: Number of sweeps
        delta_history: Convergence history
    """
    V = np.zeros(mdp.n_states)
    delta_history = []
    
    for iteration in range(max_iterations):
        delta = 0
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            
            v = V[s]
            Q = compute_action_values(mdp, V, s)  # Uses current V (in-place)
            V[s] = np.max(Q)
            delta = max(delta, abs(v - V[s]))
        
        delta_history.append(delta)
        
        if delta < theta:
            policy = extract_policy(mdp, V)
            return policy, V, iteration + 1, delta_history
    
    policy = extract_policy(mdp, V)
    return policy, V, max_iterations, delta_history


def value_iteration_standard(
    mdp: MDP,
    theta: float = 1e-6,
    max_iterations: int = 10000
) -> Tuple[np.ndarray, np.ndarray, int, List[float]]:
    """
    Value iteration with standard (two-array) updates.
    
    Returns:
        policy: Optimal policy
        V: Optimal value function
        iterations: Number of sweeps
        delta_history: Convergence history
    """
    V = np.zeros(mdp.n_states)
    delta_history = []
    
    for iteration in range(max_iterations):
        V_new = np.copy(V)  # Create copy for updates
        delta = 0
        
        for s in range(mdp.n_states):
            if s in mdp.terminal_states:
                continue
            
            Q = compute_action_values(mdp, V, s)  # Uses old V
            V_new[s] = np.max(Q)
            delta = max(delta, abs(V[s] - V_new[s]))
        
        V = V_new
        delta_history.append(delta)
        
        if delta < theta:
            policy = extract_policy(mdp, V)
            return policy, V, iteration + 1, delta_history
    
    policy = extract_policy(mdp, V)
    return policy, V, max_iterations, delta_history


print("✅ In-place and standard VI implemented")

### Comparing In-Place vs Standard

In [ ]:
print("Comparing In-Place vs Standard Updates")
print("=" * 60)

# In-place
start = time.time()
policy_ip, V_ip, iters_ip, deltas_ip = value_iteration_inplace(test_mdp)
time_ip = time.time() - start

print(f"\nIn-Place Updates:")
print(f"  Iterations: {iters_ip}")
print(f"  Time: {time_ip:.6f}s")
print(f"  Final max V: {np.max(V_ip):.6f}")

# Standard
start = time.time()
policy_std, V_std, iters_std, deltas_std = value_iteration_standard(test_mdp)
time_std = time.time() - start

print(f"\nStandard Updates:")
print(f"  Iterations: {iters_std}")
print(f"  Time: {time_std:.6f}s")
print(f"  Final max V: {np.max(V_std):.6f}")

print(f"\nComparison:")
print(f"  Iteration reduction: {100 * (1 - iters_ip / iters_std):.1f}%")
print(f"  Speedup: {time_std / time_ip:.2f}x")
print(f"  Value difference: {np.max(np.abs(V_ip - V_std)):.2e}")

# Plot convergence
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(deltas_ip, label='In-Place', linewidth=2, marker='o', markersize=4)
ax.plot(deltas_std, label='Standard', linewidth=2, marker='s', markersize=4)
ax.set_xlabel('Iteration', fontsize=12)
ax.set_ylabel('Max Value Change (Δ)', fontsize=12)
ax.set_title('In-Place vs Standard Updates: Convergence', fontsize=14, fontweight='bold')
ax.set_yscale('log')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n✅ In-place comparison complete")

## 6. Performance Benchmarking <a id='benchmark'></a>

Let's comprehensively compare all DP variants.

In [ ]:
def comprehensive_benchmark(mdp: MDP, name: str) -> Dict:
    """
    Benchmark all DP variants on a given MDP.
    
    Returns:
        results: Dictionary with performance metrics
    """
    results = {'name': name, 'algorithms': {}}
    
    algorithms = [
        ('Standard VI', lambda: value_iteration_standard(mdp)),
        ('In-Place VI', lambda: value_iteration_inplace(mdp)),
        ('Async VI (random)', lambda: async_value_iteration(mdp, state_order='random')),
        ('Prioritized Sweeping', lambda: prioritized_sweeping(mdp)),
        ('Modified PI (k=5)', lambda: modified_policy_iteration(mdp, k=5)),
        ('Modified PI (k=10)', lambda: modified_policy_iteration(mdp, k=10)),
    ]
    
    for alg_name, alg_func in algorithms:
        start = time.time()
        result = alg_func()
        elapsed = time.time() - start
        
        # Handle different return formats
        if len(result) == 4:
            policy, V, metric1, metric2 = result
            if alg_name.startswith('Modified'):
                iterations = metric2  # total sweeps
            elif alg_name == 'Prioritized Sweeping':
                iterations = metric1  # updates
            else:
                iterations = metric1  # iterations
        
        results['algorithms'][alg_name] = {
            'time': elapsed,
            'iterations': iterations,
            'V': V
        }
    
    return results


print("Running Comprehensive Benchmark")
print("=" * 70)

# Load environments
envs = [
    ('FrozenLake-4x4', gym.make('FrozenLake-v1', is_slippery=True)),
]

all_results = []
for env_name, env in envs:
    mdp = extract_mdp_from_gym(env)
    print(f"\nBenchmarking {env_name}...")
    results = comprehensive_benchmark(mdp, env_name)
    all_results.append(results)

# Display results table
print("\n" + "="*70)
print(f"{'Algorithm':<25} {'Time (s)':<12} {'Iterations':<12}")
print("="*70)

for results in all_results:
    print(f"\n{results['name']}:")
    for alg_name, metrics in results['algorithms'].items():
        print(f"  {alg_name:<23} {metrics['time']:<12.6f} {metrics['iterations']:<12}")

print("\n✅ Comprehensive benchmark complete")

## 7. Complexity Analysis <a id='complexity'></a>

Empirical validation of theoretical complexity predictions.

In [ ]:
def create_gridworld_mdp(size: int) -> MDP:
    """
    Create a square gridworld MDP for complexity testing.
    """
    n_states = size * size
    n_actions = 4  # UP, DOWN, LEFT, RIGHT
    
    P = np.zeros((n_states, n_actions, n_states))
    R = np.zeros((n_states, n_actions, n_states))
    
    def state_to_pos(s):
        return (s // size, s % size)
    
    def pos_to_state(r, c):
        return r * size + c
    
    goal_state = n_states - 1
    
    for s in range(n_states):
        r, c = state_to_pos(s)
        
        if s == goal_state:
            # Terminal state
            for a in range(n_actions):
                P[s, a, s] = 1.0
            continue
        
        # Actions: 0=UP, 1=DOWN, 2=LEFT, 3=RIGHT
        moves = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        
        for a, (dr, dc) in enumerate(moves):
            new_r, new_c = r + dr, c + dc
            
            if 0 <= new_r < size and 0 <= new_c < size:
                new_s = pos_to_state(new_r, new_c)
            else:
                new_s = s  # Hit wall, stay in place
            
            P[s, a, new_s] = 1.0
            R[s, a, new_s] = 1.0 if new_s == goal_state else -0.01
    
    return MDP(n_states, n_actions, P, R, 0.9, [goal_state])


print("Analyzing Scalability with State Space Size")
print("=" * 60)

sizes = [3, 4, 5, 6, 7, 8]
scalability_results = []

for size in sizes:
    mdp = create_gridworld_mdp(size)
    n_states = mdp.n_states
    
    print(f"\nGridWorld {size}x{size} ({n_states} states):")
    
    # Test standard VI
    start = time.time()
    policy, V, iters, _ = value_iteration_standard(mdp, theta=1e-4)
    elapsed = time.time() - start
    
    scalability_results.append({
        'size': size,
        'n_states': n_states,
        'iterations': iters,
        'time': elapsed
    })
    
    print(f"  Time: {elapsed:.4f}s, Iterations: {iters}")

print("\n✅ Scalability analysis complete")

### Visualize Complexity Scaling

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

n_states_list = [r['n_states'] for r in scalability_results]
times = [r['time'] for r in scalability_results]
iterations = [r['iterations'] for r in scalability_results]

# Time vs |S|
axes[0].plot(n_states_list, times, marker='o', linewidth=2, markersize=8, color='red')
axes[0].set_xlabel('Number of States |S|', fontsize=12)
axes[0].set_ylabel('Time (seconds)', fontsize=12)
axes[0].set_title('Value Iteration: Time Complexity', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].set_xscale('log')
axes[0].set_yscale('log')

# Iterations vs |S|
axes[1].plot(n_states_list, iterations, marker='s', linewidth=2, markersize=8, color='blue')
axes[1].set_xlabel('Number of States |S|', fontsize=12)
axes[1].set_ylabel('Iterations to Converge', fontsize=12)
axes[1].set_title('Value Iteration: Iteration Count', fontsize=14, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].set_xscale('log')

plt.tight_layout()
plt.savefig('/home/user/reinforcement-learning/static/images/2b_complexity.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Complexity visualization created")

## 8. Comprehensive Visualizations <a id='visualization'></a>

In [ ]:
# Create summary comparison plot
if all_results:
    result = all_results[0]
    alg_names = list(result['algorithms'].keys())
    times = [result['algorithms'][name]['time'] for name in alg_names]
    iters = [result['algorithms'][name]['iterations'] for name in alg_names]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Time comparison
    axes[0].barh(alg_names, times, color='steelblue', alpha=0.8)
    axes[0].set_xlabel('Time (seconds)', fontsize=12)
    axes[0].set_title('Algorithm Runtime Comparison', fontsize=14, fontweight='bold')
    axes[0].grid(True, alpha=0.3, axis='x')
    
    # Iterations comparison
    axes[1].barh(alg_names, iters, color='coral', alpha=0.8)
    axes[1].set_xlabel('Iterations/Updates', fontsize=12)
    axes[1].set_title('Algorithm Iteration Comparison', fontsize=14, fontweight='bold')
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.savefig('/home/user/reinforcement-learning/static/images/2b_algorithm_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✅ Summary visualization created")

## 9. Exercises <a id='exercises'></a>

### Exercise 1: Real-Time Dynamic Programming

Implement RTDP that updates only states visited during agent trajectories.

In [ ]:
def real_time_dp(
    mdp: MDP,
    n_episodes: int = 100,
    max_steps: int = 100
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Real-time dynamic programming.
    
    TODO: Implement this function
    
    Hint:
    - Start episodes from random states
    - Follow greedy policy
    - Update only visited states
    """
    # YOUR CODE HERE
    pass

# Test your implementation
# policy_rtdp, V_rtdp = real_time_dp(test_mdp, n_episodes=50)
# print(f"RTDP final max V: {np.max(V_rtdp):.6f}")

### Exercise 2: Gauss-Seidel Value Iteration

Implement value iteration that updates states in order and uses updated values immediately (like Gauss-Seidel method).

In [ ]:
def gauss_seidel_vi(mdp: MDP, theta: float = 1e-6) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Value iteration with Gauss-Seidel updates.
    
    TODO: Implement this function
    """
    # YOUR CODE HERE
    pass

### Exercise 3: Adaptive Modified PI

Implement modified PI where k adapts based on convergence rate.

In [ ]:
def adaptive_modified_pi(mdp: MDP) -> Tuple[np.ndarray, np.ndarray, int]:
    """
    Modified policy iteration with adaptive k.
    
    TODO: Implement this function
    
    Hints:
    - Start with small k
    - Increase k if convergence is slow
    - Decrease k if convergence is fast
    """
    # YOUR CODE HERE
    pass

### Exercise 4: Comparative Analysis

Create a larger benchmark comparing all methods (standard, async, prioritized, modified) on FrozenLake 8x8.

In [ ]:
# TODO: Benchmark on FrozenLake-v1 8x8
# YOUR CODE HERE

# env_8x8 = gym.make('FrozenLake-v1', map_name="8x8", is_slippery=True)
# mdp_8x8 = extract_mdp_from_gym(env_8x8)
# results_8x8 = comprehensive_benchmark(mdp_8x8, "FrozenLake-8x8")
# ...

---

## Summary

In this notebook, you have:

1. ✅ **Implemented Asynchronous VI** - Flexible state update ordering
2. ✅ **Implemented Prioritized Sweeping** - Priority-based computation focusing
3. ✅ **Implemented Modified PI** - Interpolating between VI and PI
4. ✅ **Implemented In-Place Updates** - Memory-efficient value iteration
5. ✅ **Benchmarked Performance** - Comprehensive comparison across algorithms
6. ✅ **Analyzed Complexity** - Empirical validation of theoretical scaling

### Key Insights

1. **Asynchronous methods** provide flexibility without sacrificing convergence guarantees
2. **Prioritized sweeping** can dramatically reduce computation by focusing on important states
3. **Modified PI** offers tunable trade-off between VI and PI
4. **In-place updates** typically converge faster and use less memory
5. **No single algorithm dominates** - best choice depends on problem structure

### Algorithm Selection Guide

**Use Standard VI when**:
- Simple, predictable behavior needed
- Parallelization is important

**Use Prioritized Sweeping when**:
- Sparse state space with bottlenecks
- Minimizing updates is critical

**Use Modified PI when**:
- Want faster convergence than VI
- Can't afford full policy evaluation

**Use In-Place when**:
- Memory is constrained
- Faster convergence preferred

### Transition to Model-Free RL

Dynamic programming provides the foundation, but requires a full model. In **Lesson 3**, we'll explore:
- **Monte Carlo methods**: Learning from complete episodes
- **Temporal Difference learning**: Bootstrap + sampling
- **Model-free control**: Q-learning and SARSA

These methods work without knowing $p(s',r|s,a)$, making them applicable to real-world problems.

---

**Lesson 2b Complete!** 🎉

**Foundation Phase (Lessons 0-2) Complete!**

You now have comprehensive knowledge of dynamic programming methods and are ready to move into model-free reinforcement learning.